# Capstone — Content Refresh Opportunity Scoring

**Research paper companion notebook for Lane 2.** This work asks whether a transparent score can help an editor choose which content items to review first for a possible refresh. The output is decision support and a ranked review queue, not an autonomous editing system.

## 1. Question

### Research question

**Among pseudonymized content items in the FlyRank starter snapshot, can a transparent model rank a short list of pages that are worth human review for possible content refresh?**

The decision is which pages an SEO or content analyst should inspect first. The action is a bounded review of title/snippet alignment, freshness, depth, and intent—not an automatic rewrite. A wrong recommendation has two costs: editorial time can be spent on a page that did not need attention, or a genuinely useful review opportunity can be missed. ML can help because the signals interact across exposure, position, CTR, freshness, content structure, and intent; the score can make that prioritization explicit and repeatable.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO = Path('/home/ubuntu/flyrank-ml-internship-starter')
DATA = REPO / 'data/raw/content_refresh_anonymized.csv'
OUT = REPO / 'work/outputs'
FIG = REPO / 'work/figures'
FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA)
print(f'Starter snapshot: {len(df):,} rows | {df.client_id.nunique()} pseudonymous clients | {df.shape[1]} columns')
print('Unit of analysis: one pseudonymized content item at the trailing-90-day snapshot.')
print('Lane: Content Refresh Opportunity Scoring | task: ranking/scoring | human action: review first')

## 2. Data

The executed analysis uses the repository’s **30,000-row starter release**, `data/raw/content_refresh_anonymized.csv`, with one row per pseudonymized content item and trailing-90-day aggregate metrics. The repository documentation describes a separate Hugging Face warehouse release with daily performance, content, client, and query tables spanning 2025-01-27 to 2026-06-30; this capstone does **not** claim to have trained on that 79M-row warehouse. The warehouse is excluded here because the submitted lane was developed and validated on the starter snapshot, and mixing grains or final-month outcome windows would risk leakage.

Excluded from model inputs are pseudonymous IDs, `trend_direction`, `trend_pct`, `is_declining_label`, future-window fields, and product-decision fields. Rates are interpreted using the data dictionary convention: values such as `ctr = 0.76` mean 0.76%, not 76%. `avg_position = 0` is treated as no data rather than rank zero.

In [ ]:
summary = pd.DataFrame({
    'quantity': ['rows','pseudonymous clients','target proxy rate','median impressions_90d','median days_since_last_update'],
    'value': [len(df), df.client_id.nunique(), (df.trend_direction == 'down').mean(), df.impressions_90d.median(), df.days_since_last_update.median()]
})
display(summary)

## 3. Methodology

The target is an evaluation proxy: `target = 1` when `trend_direction == "down"`. It is not a claim that a page requires an edit, and it is never used as an input. The learned model is Logistic Regression with training-only median/mode imputation, numeric scaling, and one-hot encoding. The Week-4 baseline is the transparent rule `visible × page_one_two × low_ctr × log1p(impressions_90d)`.

The primary validation design is a grouped holdout by `client_id`, with no client appearing in both train and test. ML-07 also compared this with a random row split and showed why the grouped result is more conservative. The leakage audit excludes label-derived trend fields, future-window fields, IDs, and decision flags; a deliberate target-copy probe produced a perfect score as expected and was excluded from the final model.

In [ ]:
ml06 = json.loads((OUT / 'ml06_model_metrics.json').read_text())
ml07 = json.loads((OUT / 'ml07_validation_audit_metrics.json').read_text())
features = ml06['features']
forbidden = {'trend_direction','trend_pct','is_declining_label','target','client_id','content_id'}
print('Feature count:', len(features))
print('Forbidden overlap:', set(features) & forbidden)
print('Grouped client overlap:', ml07['after_grouped_client_split']['train_clients'], 'train clients vs', ml07['after_grouped_client_split']['test_clients'], 'test clients; overlap was checked as zero in ML-07.')
assert not (set(features) & forbidden)

## 4. Results (vs baseline)

The model and baseline are measured on the same held-out client groups and with the same ranking metric. The grouped test base rate is shown beside the ranking results. Precision@10 and Precision@50 are the operational metrics because the intended action is a short human review queue; average precision and ROC AUC are secondary diagnostics.

In [ ]:
comparison = pd.DataFrame([ml06['baseline'], ml06['model']])
comparison['test_base_rate'] = ml06['test_base_rate']
display(comparison.round(4))
plot_cols = ['precision_at_10','precision_at_50','precision_at_100']
plot = comparison.set_index('method')[plot_cols]
ax = plot.T.plot(kind='bar', figsize=(9,5), color=['#9aa7b2','#2f6f8f'])
ax.set_title('Held-out client evaluation: model versus Week-4 rule')
ax.set_ylabel('Measured precision')
ax.set_xlabel('Ranking cutoff')
ax.set_ylim(0,1)
ax.legend(title='Method', loc='lower right')
fig = ax.get_figure(); fig.tight_layout()
fig_path = FIG / 'capstone_model_vs_baseline.png'
fig.savefig(fig_path, dpi=160, bbox_inches='tight')
plt.show()
print('Chart saved to', fig_path)

On the grouped holdout, the Logistic Regression ranking measured **0.80 Precision@10** and **0.58 Precision@50**, compared with **0.40** and **0.42** for the Week-4 rule in the prior controlled comparison. The model therefore ranks more positive proxy outcomes near the top in this measured split, but the ROC AUC of approximately **0.55** is modest; this is a prioritization result, not a causal recovery estimate.

## 5. Limitations and honest framing

This is a cross-sectional starter snapshot with a proxy label, not a prospective experiment. It does not estimate the effect of editing a page, does not prove that age or CTR causes decline, and does not guarantee transfer to new clients, future months, search engines, or business outcomes. The grouped holdout reduces client-mix leakage, but a time-aware prospective evaluation and a pre-specified follow-up window are still needed.

The queue should remain human-reviewed. Editors should verify the page, intent, SERP context, recent changes, business importance, and legal or brand constraints. A low score is not a quality judgment, and a high score is not permission to publish, delete, redirect, or mass-edit content.

## 6. Ranked recommendations

The action playbook converts the ranking into bounded review actions. `R1_VISIBLE_STALE_LOW_CTR` is reviewed first because it combines exposure, visibility, low CTR, and staleness; the reviewer checks title/snippet alignment before considering a refresh. `R2_VISIBLE_LOW_CTR` is a snippet and intent review. `R3` and `R4` cover thin or stale content with a bounded brief. `R6` and `R7` are monitoring/defer actions when evidence or exposure is weaker.

The queue is regenerated by ML-08 at `work/outputs/ranked_action_playbook.csv`; it is intentionally excluded from Git because it is a data file. Each row has one reason code and one recommended action. No action is automatically applied.

In [ ]:
queue = pd.read_csv(OUT / 'ranked_action_playbook.csv')
public_top10 = queue.head(10).drop(columns=['content_id','client_id'], errors='ignore')
display(public_top10[['queue_rank','review_priority','model_priority_score','reason_code','archetype','recommended_action','impressions_90d','ctr','avg_position','days_since_last_update']].round(4))
print('Queue rows:', len(queue), '| reason codes:', queue.reason_code.nunique())

## 7. Artifacts the paper embeds

The deployed page embeds the model-versus-baseline chart at `docs/assets/capstone_model_vs_baseline.png` and the reason-code chart at `docs/assets/w07_reason_code_mix.png`. The main reproducibility receipts are `work/outputs/ml06_model_metrics.json`, `work/outputs/ml07_validation_audit_metrics.json`, and `work/outputs/ml08_action_playbook_metrics.json`. The notebook links below point to the public repository paths so a reader can inspect the analysis.

In [ ]:
artifacts = pd.DataFrame([
    ['ML-05 baseline', 'work/notebooks/w04_baseline_score.ipynb'],
    ['ML-06 model', 'work/notebooks/w05_model.ipynb'],
    ['ML-07 validation audit', 'work/notebooks/w06_validation_audit.ipynb'],
    ['ML-08 action playbook', 'work/notebooks/w07_action_playbook.ipynb'],
    ['Results chart', 'work/figures/capstone_model_vs_baseline.png'],
    ['Action mix chart', 'work/figures/w07_reason_code_mix.png'],
], columns=['artifact','repository_path'])
display(artifacts)

## Reproducibility

The repository is [Ahmedosrf/flyrank-ml-internship-ahmedosrf](https://github.com/Ahmedosrf/flyrank-ml-internship-ahmedosrf). The analysis uses seed 42 where stochastic splitting or modeling is involved. Run the notebooks in order from `work/notebooks/` in an environment with the repository dependencies and the included starter data. The queue and metrics receipts are regenerated by the notebooks; no token, private query, client name, or URL is required.

## Acknowledgments & data credit

Built on the **FlyRank ML Internship dataset**. Data source: [FlyRank](https://flyrank.ai). The dataset is used for educational research and this page reports only public-safe, pseudonymized, aggregate findings.

## ML-12 communication package

### Five-minute demo outline

1. State the Lane 2 decision: which pages deserve human review first.
2. Show the transparent Week-4 baseline and why it is useful but limited.
3. Show the Logistic Regression ranking and the grouped-by-client validation design.
4. Show the model-versus-baseline Precision@10/50 chart and explain the modest ROC AUC.
5. Open the action playbook: reason codes, human-review rules, monitoring triggers, and no-go cases.

### Social-post cut

I built a transparent content-refresh opportunity scorer on FlyRank’s anonymized starter dataset. A grouped client holdout measured higher short-queue precision for the Logistic Regression ranking than the Week-4 rule, while the modest ROC AUC kept the conclusion appropriately cautious. The result is a human-reviewed action queue—not an automatic content editor.

### Employer-facing summary

I framed content refresh as a ranking problem, built a transparent baseline and Logistic Regression model, and validated the comparison with a client-grouped holdout. The model measured 0.80 Precision@10 and 0.58 Precision@50 on the grouped test split versus 0.40 and 0.42 for the Week-4 rule. I converted the result into a reason-coded, human-reviewed playbook with leakage checks, limits, and monitoring triggers.

## Self-check

- [x] All required paper sections are filled: abstract, introduction/problem, data, methodology, results, limitations, ranked recommendations, reproducibility, and acknowledgments/data credit.
- [x] The notebook runs top to bottom and creates the reusable results chart.
- [x] The data source is credited with the FlyRank link.
- [x] Claims use observed, measured, directional, and decision-support language.
- [x] No client names, private queries, tokens, or raw URLs appear in the paper artifacts.
- [x] The ML-12 demo outline, social-post cut, and employer-facing summary are included.
